In [1]:
import multiprocessing
import os
import re
import csv
import random
import torch
import pandas as pd
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.chat_models import ChatLlamaCpp

In [2]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [3]:
# Silence llama.cpp
os.environ["LLAMA_LOG_LEVEL"] = "ERROR" 

In [4]:
def generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, filename, dataset):
  # generate and save notes

    def parse_and_clean_reports(results):
        # parse and clean reports from openai completions
        reports = []

        for result in results:
            text = result.get("output", "")
            text = text.replace('\n', ' ')
            pattern = r'\*\*\*|\s(?=\d{1,2}[\.,]{1,2}\s)' # split by *** or numbers followed by . or , 
            # Split by *** or quotes
            splits = re.split(pattern, text)
            for item in splits:
                if item:
                    cleaned = re.sub(r'^\s*\d{1,2}[\.,]{1,3}\s*', '', item)
                    cleaned = cleaned.strip()
                    # Keep only items with letters, no colon or quote
                    if cleaned and re.search(r'[a-zA-Z]', cleaned):  # keep only if contains letters
                        reports.append(cleaned)

        return reports

    def save_reports(reports, filename):
        df = pd.DataFrame(reports, columns=['report'])
        try:
            df.to_csv(filename, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', lineterminator='\n')
            print(f"Reports saved successfully to {filename}")
        except Exception as e:
            print(f"Failed to save reports: {str(e)}")

    data_dir = f'./data/{dataset}'  # update if needed
    os.makedirs(data_dir, exist_ok=True)
    report_filepath = os.path.join(data_dir, f'{filename}.csv')

    # initialize local model
    llm = ChatLlamaCpp(
    model_path=model,
    temperature=temperature,
    n_ctx=10000,
    n_gpu_layers=8,
    n_batch=300,
    max_tokens=512,
    n_threads=max(1, multiprocessing.cpu_count() - 1),
    repeat_penalty=1.5,
    top_p=0.5,
    verbose=False,
    )

    # create prompts
    role_prompt = PromptTemplate(template=system_role_prompt['message'], input_variables=system_role_prompt['inputs'])
    note_prompt = PromptTemplate(template=note_query_prompt['message'], input_variables=note_query_prompt['inputs'])

    # create llmchains
    role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")
    note_chain = LLMChain(llm=llm, prompt=note_prompt, output_key = "output")
    # create sequentialchain
    sequential_chain = SequentialChain(
        chains=[role_chain, note_chain], input_variables = (system_role_prompt['inputs'] + note_query_prompt['inputs']), output_variables = ["output"]
    )

    # generate a number of different responses
    results = []
    for _ in range(completions):
        response = sequential_chain.invoke(input_data)
        results.append(response)

    # clean results
    results = parse_and_clean_reports(results)
    # save results
    save_reports(results, report_filepath)


In [5]:
system_role_prompt = {'message':
                      '''
                      You are a specialist in generating fictitious data for natural language processing projects in healthcare.
                      You speak the language of a nurse in an {nationality} nursing home. Namely, you speak {language}.
                      ''' ,
                      'inputs':["nationality", "language"]}

note_query_prompt = {'message':
                      '''
                      This is an example of a nurse note for a patient in a day: "{example_note}"

                      Other reports may include: {topic_keywords}

                      Most reports are about everyday things, so not everything is a serious incident.

                      Make up {number_of_reports} such reports for {number_of_reports} residents with {needs} palliative care needs. Return only the reports, with each report separated by "***" and nothing else. Vary the sentence structure and style.
                      ''' ,
                      'inputs':["example_note", "topic_keywords", "number_of_reports", "needs"]}

In [6]:
# Get input prompt data.
df = pd.read_csv('../promptDataPreparation/promptDataPreparation.csv')
fake_notes = pd.read_excel('./fake_notes.xlsx')

In [7]:
matave_topic_keywords = df[df['topic_model'] == 'MATAVE']['topic_keywords'].tolist()[0]
lda_topic_keywords = df[df['topic_model'] == 'LDA']['topic_keywords'].tolist()[0]

topic_model_dict = {'MATAVE'}

In [8]:
completions = 25
model = './models/Phi-4-mini-instruct.Q8_0.gguf'
temperature = 1.1

for topic_model in ['MATAVE', 'LDA']:
    topic_keywords = df[df['topic_model'] == topic_model]['topic_keywords'].tolist()[0]
    for index, row in fake_notes.iterrows():
        input_data = {'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': row.Note, 'topic_keywords': topic_keywords, 'number_of_reports': 25, 'needs': row.Needs}
        print(input_data)
        generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, f"{index % 5}_{row.Needs}", topic_model)

{'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': 'John was assisted with a shower this morning. He had his lunch in the canteen, he ate half a bowl of soup, half a chicken salad and a full bowl of ice cream along with 2 cups of tea. Johns’ sister Mary came to see him today and he was in good form. ', 'topic_keywords': 'Other topic keywords: restaurant nil bright content intake voice meal club baseline new\nOther topic keywords: bed nocte issue overnight reach safe sleep comfortably administer bell\nOther topic keywords: charted compliant toilette change nil self assisted maintain meds adls\nOther topic keywords: accordingly skin tolerate attend abdomen abnormality abx accept accompany acre', 'number_of_reports': 25, 'needs': 'met'}


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
/var/folders/12/p5c0vdcj3yx9xb69fcd0n3j80000gn/T/ipykernel_15315/3536857066.py:55: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")


Reports saved successfully to ./data/MATAVE/0_met.csv
{'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': 'Sarah had a GP appointment this morning and the GP was happy with her mobility progress. She has been advised to continue with her physiotherapy, but her pain relief has been reduced as her pain has been well controlled. The GP hopes that she will be able to walk without crutches within the next month.', 'topic_keywords': 'Other topic keywords: restaurant nil bright content intake voice meal club baseline new\nOther topic keywords: bed nocte issue overnight reach safe sleep comfortably administer bell\nOther topic keywords: charted compliant toilette change nil self assisted maintain meds adls\nOther topic keywords: accordingly skin tolerate attend abdomen abnormality abx accept accompany acre', 'number_of_reports': 25, 'needs': 'met'}


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Reports saved successfully to ./data/MATAVE/1_met.csv
{'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': 'Michael showered independently this morning. It was his birthday today, so his family came to visit. His wife organised a sing song in the social room. Michael was in very good spirits and sang a few songs himself. ', 'topic_keywords': 'Other topic keywords: restaurant nil bright content intake voice meal club baseline new\nOther topic keywords: bed nocte issue overnight reach safe sleep comfortably administer bell\nOther topic keywords: charted compliant toilette change nil self assisted maintain meds adls\nOther topic keywords: accordingly skin tolerate attend abdomen abnormality abx accept accompany acre', 'number_of_reports': 25, 'needs': 'met'}


...

Reports saved successfully to ./data/LDA/2_unmet.csv
{'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': 'Louise was assisted with a wash this morning. It was discovered that she had a 5cm lesion to her left buttock that contains slough and appears infected. A swab was taken and a wound chart commenced. Inadine and Leukomed (wound dressings) in situ at present. Awaiting review from Tissue Viability Nurse.', 'topic_keywords': 'Other topic keywords: care resident plan give assist complaint medication day form morning\nOther topic keywords: settle give medication resident sleep voice night issue bed comfortably\nOther topic keywords: resident nil concern med good assist new care appear give\nOther topic keywords: resident safety check settle need maintain night have adls meds\nOther topic keywords: resident sleep night check continue medication settle care safety nocte\nOther topic keywords: resident care med chart need assist appear attend form take\nOther topic keywo

llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Reports saved successfully to ./data/LDA/3_unmet.csv
{'nationality': 'Irish', 'language': 'Hiberno-English', 'example_note': 'Michelle was assisted with her breakfast this morning by one of the healthcare assistants. The healthcare assistant expressed her concern that Michelle’s swallow seems to be worsening. A referral has been made for the Speech and Language Therapist to review. For now, we have placed Michelle on a level 5 diet and level 2 fluids until she is reviewed.', 'topic_keywords': 'Other topic keywords: care resident plan give assist complaint medication day form morning\nOther topic keywords: settle give medication resident sleep voice night issue bed comfortably\nOther topic keywords: resident nil concern med good assist new care appear give\nOther topic keywords: resident safety check settle need maintain night have adls meds\nOther topic keywords: resident sleep night check continue medication settle care safety nocte\nOther topic keywords: resident care med chart need 

llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Reports saved successfully to ./data/LDA/4_unmet.csv
